# AR NutriLens - YOLOv8 Instance Segmentation Training (24 Classes - 100 Epochs)
### Huấn luyện mô hình bóc tách vùng Món ăn Việt Nam + Nguyên liệu đơn lẻ (Trứng luộc, Cà rốt, Ức gà...)

> **Lưu ý:** Trước khi chạy, chọn **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# 1. Kiểm tra kết nối GPU NVIDIA
!nvidia-smi

# 2. Cài đặt Ultralytics và Roboflow mới nhất
!pip install -q ultralytics roboflow

In [ ]:
# 3. Tải và giải nén Dataset 14 món ăn Việt Nam nền tảng
!wget -q https://huggingface.co/datasets/HoagMin/Vietnamese_StreetFood_Dataset/resolve/main/Vietnamese_StreetFood_14Class.zip
!unzip -q Vietnamese_StreetFood_14Class.zip -d /content/dataset
print("Tải và giải nén dữ liệu 14 món Việt Nam hoàn tất!")

In [ ]:
# 4. Chuyển đổi nhãn Bounding Box sang chuẩn Instance Segmentation Polygon
import os, glob

converted_count = 0
for split in ['train', 'valid', 'test']:
    lbl_dir = f'/content/dataset/{split}/labels'
    for p in glob.glob(f'{lbl_dir}/*.txt'):
        lines = []
        with open(p, 'r') as f:
            for l in f:
                parts = l.strip().split()
                if len(parts) == 5:
                    c, xc, yc, w, h = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1, y1 = max(0, min(1, xc - w/2)), max(0, min(1, yc - h/2))
                    x2, y2 = max(0, min(1, xc + w/2)), max(0, min(1, yc - h/2))
                    x3, y3 = max(0, min(1, xc + w/2)), max(0, min(1, yc + h/2))
                    x4, y4 = max(0, min(1, xc - w/2)), max(0, min(1, yc + h/2))
                    lines.append(f"{c} {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}\n")
                    converted_count += 1
                else:
                    lines.append(l)
        with open(p, 'w') as f:
            f.writelines(lines)
print(f"Chuyển đổi thành công {converted_count} nhãn sang chuẩn Polygon Mask!")

In [ ]:
# 5. Tạo cấu hình data_seg.yaml cho 24 Classes (14 Món Việt + 10 Nguyên liệu rời)
yaml_content = """
path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 24
names:
  0: BANH_CUON
  1: BANH_GIO
  2: BANH_MI
  3: BUN_BO_HUE
  4: BUN_CHA_HA_NOI
  5: BUN_DAU_MAM_TOM
  6: BUN_THIT_NUONG
  7: CHAO_LONG
  8: COM_DEP
  9: BROKEN_RICE
  10: COM_RANG
  11: BEEF_PHO
  12: GOI_CUON
  13: XOI
  14: BOILED_EGG
  15: CARROT
  16: CHICKEN_BREAST
  17: BROCCOLI
  18: CUCUMBER
  19: SWEET_POTATO
  20: TOFU_STEAMED
  21: SHRIMP_BOILED
  22: RICE_WHITE
  23: PORK_GRILLED
"""
with open("/content/dataset/data_seg.yaml", "w") as f:
    f.write(yaml_content)
print("Đã lưu cấu hình /content/dataset/data_seg.yaml cho 24 lớp!")

In [ ]:
# 6. Tiến hành huấn luyện 100 Epochs với YOLOv8n-seg trên GPU T4
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
results = model.train(
    data="/content/dataset/data_seg.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    amp=True,
    patience=25,
    name="nutrilens_vietfood_seg_24class"
)
print("Huấn luyện 100 Epochs hoàn tất!")

In [ ]:
# 7. Tự động tải file trọng số tốt nhất (best.pt) về máy
from google.colab import files
best_pt_path = f"{results.save_dir}/weights/best.pt"
print(f"Đang tải file trọng số: {best_pt_path}")
files.download(best_pt_path)
# Sau khi tải về: đổi tên hoặc chép đè vào thư mục ai-service/weights/yolov8_vietfood_seg.pt